In [1]:
import pandas as pd
import numpy as np

TARGETS = ["Y"]

SETS = ["Train_1", "Train_2", "Val_1", "Val_2", "Test_1", "Test_2", "LSG_1", "LSG_2"]

results = pd.read_excel("resultados.xlsx")


In [2]:

results = pd.read_excel("resultados.xlsx")

best_models_tables = {}

N = 10  # top modelos

for target in TARGETS:

    # 🔹 nomes das colunas
    r2_cols = {s: f"R2_{s}_{target}" for s in SETS}

    # 🔹 garantir que só usamos colunas existentes
    valid_r2 = {k: v for k, v in r2_cols.items() if v in results.columns}


    df = results.copy()# 🔹 pegar todas as colunas de R2 válidas
    r2_all_cols = list(valid_r2.values())

    # 🔹 remover linhas onde QUALQUER R2 < 0
    df = df[(df[r2_all_cols] >= 0).all(axis=1)]

    # =========================
    # 🔹 MÉDIAS POR GRUPO
    # =========================
    
    train_cols = [v for k, v in valid_r2.items() if "Train" in k]
    test_cols  = [v for k, v in valid_r2.items() if "Test" in k]
    val_cols   = [v for k, v in valid_r2.items() if "Val" in k]
    test2_cols  = [v for k, v in valid_r2.items() if "LSG" in k]


    df["R2_train_mean"] = df[train_cols].mean(axis=1)
    df["R2_test_mean"]  = df[test_cols].mean(axis=1)
    df["R2_val"]        = df[val_cols].mean(axis=1)
    df["R2_test2_mean"]        = df[test2_cols].mean(axis=1)

    # =========================
    # 🔹 SCORE
    # =========================
    
    w_val = 0.3
    w_test = 0.2
    w_test2 = 0.3
    w_train = 0.2

    metric_cols = list(r2_cols.values())
    df["R2_std"] = df[metric_cols].std(axis=1)

    df["Score"] = (
        w_val * df["R2_val"] +
        w_test * df["R2_test_mean"] +
        w_test2 * df["R2_test2_mean"] +
        w_train * df["R2_train_mean"]
        - 0.1 * df["R2_std"]   # penaliza inconsistência
    )

    # =========================
    # 🔹 ORDENAÇÃO
    # =========================
    
    df_sorted = df.sort_values(by="Score", ascending=False)

    best_models_tables[target] = df_sorted

    # =========================
    # 🔹 TOP N RESUMO
    # =========================
    
    print(f"\n🏆 TOP {N} MODELOS - {target}")
    display(df_sorted[
        ["model", "Neurons", "R2_val", "R2_test_mean", "R2_test2_mean", "R2_train_mean", "Score"]
    ].head(N))

    top_df = df_sorted.head(N).copy()

    # 🔹 organizar colunas
    metric_cols = []

    for s in SETS:
        if s in valid_r2:
            metric_cols.append(valid_r2[s])

    final_cols = ["model", "Neurons"] + metric_cols + [
        "R2_val", "R2_test_mean", "R2_test2_mean", "R2_train_mean", "Score"
    ]

    final_table = top_df[final_cols]

    print(f"\n📊 MÉTRICAS COMPLETAS - TOP {N} ({target})")
    display(final_table)


🏆 TOP 10 MODELOS - Y


,model,Neurons,R2_val,R2_test_mean,R2_test2_mean,R2_train_mean,Score
228,model_arch16-8-4_r0.9_Ld0.3_Lp0.7_seed8792,"[16, 8, 4]",0.925289,0.761581,0.210261,0.990122,0.656069
133,model_arch16-8_r0.01_Ld0.7_Lp0.3_seed8792,"[16, 8]",0.848034,0.715087,0.277883,0.987870,0.648466
225,model_arch16-8-4_r0.9_Ld0.3_Lp0.7_seed4602,"[16, 8, 4]",0.762790,0.843091,0.270056,0.972868,0.642217
153,model_arch32-16_r0.01_Ld0.7_Lp0.3_seed8792,"[32, 16]",0.873836,0.652239,0.256604,0.972847,0.632010
139,model_arch16-8_r0.9_Ld0.7_Lp0.3_seed547,"[16, 8]",0.895531,0.696301,0.196304,0.984575,0.630820
113,model_arch8-4_r0.01_Ld0.7_Lp0.3_seed8792,"[8, 4]",0.949649,0.647690,0.162220,0.933203,0.614861
2,model_arch16_r0.01_Ld0.3_Lp0.7_seed6756,[16],0.891079,0.564402,0.222627,0.988269,0.609164
3,model_arch16_r0.01_Ld0.3_Lp0.7_seed8792,[16],0.906921,0.377306,0.370866,0.889329,0.607482
49,model_arch64_r0.9_Ld0.3_Lp0.7_seed547,[64],0.918335,0.418223,0.220775,0.987923,0.587630
165,model_arch64-32_r0.9_Ld0.3_Lp0.7_seed4602,"[64, 32]",0.844112,0.524612,0.225296,0.974860,0.587467



📊 MÉTRICAS COMPLETAS - TOP 10 (Y)


,model,Neurons,R2_Train_1_Y,R2_Train_2_Y,R2_Val_1_Y,R2_Val_2_Y,R2_Test_1_Y,R2_Test_2_Y,R2_LSG_1_Y,R2_LSG_2_Y,R2_val,R2_test_mean,R2_test2_mean,R2_train_mean,Score
228,model_arch16-8-4_r0.9_Ld0.3_Lp0.7_seed8792,"[16, 8, 4]",0.991936,0.988308,0.885412,0.965166,0.568494,0.954668,0.102258,0.318264,0.925289,0.761581,0.210261,0.990122,0.656069
133,model_arch16-8_r0.01_Ld0.7_Lp0.3_seed8792,"[16, 8]",0.991225,0.984515,0.706464,0.989604,0.634643,0.795530,0.219522,0.336244,0.848034,0.715087,0.277883,0.987870,0.648466
225,model_arch16-8-4_r0.9_Ld0.3_Lp0.7_seed4602,"[16, 8, 4]",0.963584,0.982151,0.573604,0.951976,0.727961,0.958221,0.250736,0.289375,0.762790,0.843091,0.270056,0.972868,0.642217
153,model_arch32-16_r0.01_Ld0.7_Lp0.3_seed8792,"[32, 16]",0.979346,0.966348,0.784396,0.963277,0.430898,0.873579,0.218666,0.294541,0.873836,0.652239,0.256604,0.972847,0.632010
139,model_arch16-8_r0.9_Ld0.7_Lp0.3_seed547,"[16, 8]",0.981661,0.987489,0.857857,0.933204,0.634894,0.757708,0.164407,0.228200,0.895531,0.696301,0.196304,0.984575,0.630820
113,model_arch8-4_r0.01_Ld0.7_Lp0.3_seed8792,"[8, 4]",0.939792,0.926614,0.949584,0.949714,0.519002,0.776378,0.103927,0.220513,0.949649,0.647690,0.162220,0.933203,0.614861
2,model_arch16_r0.01_Ld0.3_Lp0.7_seed6756,[16],0.987244,0.989294,0.823572,0.958585,0.294866,0.833938,0.177828,0.267426,0.891079,0.564402,0.222627,0.988269,0.609164
3,model_arch16_r0.01_Ld0.3_Lp0.7_seed8792,[16],0.985222,0.793436,0.934391,0.879452,0.492656,0.261957,0.370278,0.371454,0.906921,0.377306,0.370866,0.889329,0.607482
49,model_arch64_r0.9_Ld0.3_Lp0.7_seed547,[64],0.984981,0.990865,0.946400,0.890271,0.309253,0.527194,0.250115,0.191435,0.918335,0.418223,0.220775,0.987923,0.587630
165,model_arch64-32_r0.9_Ld0.3_Lp0.7_seed4602,"[64, 32]",0.984184,0.965536,0.715699,0.972525,0.388795,0.660429,0.111922,0.338670,0.844112,0.524612,0.225296,0.974860,0.587467


In [3]:
final_table.to_excel("BestModels.xlsx")